# Test: Heatmap Pin Exclusion Mask & Via-in-Pad Boost

This notebook tests the two new fixes:
1. **Pin Exclusion Mask** — zeros out heatmap areas near other-net pads
2. **Via-in-Pad Boost** — boosts via probability at the current net's own pads

In [ ]:
# Setup: clone or update repo
import os

REPO_DIR = 'Router'
if os.path.exists(REPO_DIR):
    print('Repo exists, pulling latest...')
    !cd {REPO_DIR} && git fetch --all && git reset --hard origin/master
else:
    print('Cloning repo...')
    !git clone https://github.com/Klutzhehe/Router.git

%cd {REPO_DIR}
!pip install -e . -q
!pip install matplotlib numpy pyyaml gymnasium -q
print('Setup done!')

In [ ]:
import sys
import os

# Make sure we're in the repo root
if os.path.basename(os.getcwd()) != 'Router':
    if os.path.exists('Router'):
        os.chdir('Router')
sys.path.insert(0, '.')

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches

from pcb_router.data.board_generator import BoardGenerator, BoardConfig
from pcb_router.env.board_state import BoardState

print('Imports OK')

## 1. Generate a test board

In [ ]:
bg = BoardGenerator()
bc = BoardConfig(
    board_width=150, board_height=150,
    num_nets=5, num_layers=2,
    num_components=4, obstacle_density=0.05,
    seed=42
)
board = bg.generate(bc)
bs = BoardState(board)

print(f'Board: {board.width}x{board.height}, {len(board.nets)} nets, {len(board.pins)} pins')
for net in board.nets:
    pin_locs = [(board.pins[pid].global_x, board.pins[pid].global_y) for pid in net.pin_ids]
    print(f'  Net {net.id} ({net.name}): pins at {pin_locs}')

## 2. Test Pin Exclusion Mask

Set net 0 as current, then check that:
- Net 0's own pads are **NOT** masked (mask = 1.0)
- Other nets' pads **ARE** masked (mask = 0.0)

In [ ]:
# Pick the first net as the "current" net being routed
current_net = board.nets[0]
bs.set_current_net(current_net.id)

# Get exclusion mask for layer 0
mask_L0 = bs.get_pin_exclusion_mask(layer=0)

print(f'Exclusion mask shape: {mask_L0.shape}')
print(f'  Blocked cells (0.0): {(mask_L0 == 0).sum()}')
print(f'  Open cells (1.0):    {(mask_L0 == 1).sum()}')
print()

# Check own pads are NOT blocked
print('--- Current net pads (should be 1.0 = open) ---')
for pid in current_net.pin_ids:
    pin = board.pins[pid]
    val = mask_L0[pin.global_y, pin.global_x]
    status = 'PASS' if val == 1.0 else 'FAIL'
    print(f'  [{status}] Net {current_net.id} pin at ({pin.global_x}, {pin.global_y}): mask = {val:.2f}')

# Check other nets' pads ARE blocked
print()
print('--- Other net pads (should be 0.0 = blocked) ---')
for net in board.nets[1:]:
    for pid in net.pin_ids:
        pin = board.pins[pid]
        val = mask_L0[pin.global_y, pin.global_x]
        status = 'PASS' if val == 0.0 else 'FAIL'
        print(f'  [{status}] Net {net.id} pin at ({pin.global_x}, {pin.global_y}): mask = {val:.2f}')

## 3. Test Via-in-Pad Boost

In [ ]:
boost = bs.get_via_in_pad_boost()

print(f'Via boost shape: {boost.shape}')
print(f'  Boosted cells (>0): {(boost > 0).sum()}')
print()

# Own pads should be boosted
print('--- Current net pads (should have boost = 1.0) ---')
for pid in current_net.pin_ids:
    pin = board.pins[pid]
    val = boost[pin.global_y, pin.global_x]
    status = 'PASS' if val == 1.0 else 'FAIL'
    print(f'  [{status}] Net {current_net.id} pad at ({pin.global_x}, {pin.global_y}): boost = {val:.2f}')

# Other pads should NOT be boosted
print()
print('--- Other net pads (should have boost = 0.0) ---')
for net in board.nets[1:]:
    for pid in net.pin_ids:
        pin = board.pins[pid]
        val = boost[pin.global_y, pin.global_x]
        status = 'PASS' if val == 0.0 else 'FAIL'
        print(f'  [{status}] Net {net.id} pad at ({pin.global_x}, {pin.global_y}): boost = {val:.2f}')

## 4. Visualize: Before vs After Heatmap Filtering

Shows what happens when we apply the pin exclusion mask to a uniform heatmap.

In [ ]:
# Create a fake "model output" heatmap - uniform 0.5 everywhere
raw_heatmap = np.ones((board.num_layers, board.height, board.width), dtype=np.float32) * 0.5

# Apply pin exclusion mask
filtered_heatmap = raw_heatmap.copy()
for l in range(board.num_layers):
    pin_mask = bs.get_pin_exclusion_mask(l)
    filtered_heatmap[l] *= pin_mask

# Via prob: uniform 0.3 + via-in-pad boost
raw_via = np.ones((board.height, board.width), dtype=np.float32) * 0.3
boosted_via = np.clip(raw_via + boost, 0.0, 1.0)

# --- Plot ---
fig, axes = plt.subplots(2, 2, figsize=(14, 12))
fig.patch.set_facecolor('#111222')
fig.suptitle(f'Heatmap Filtering for Net {current_net.id} ({current_net.name})',
             color='white', fontsize=16, y=0.98)

titles = ['Raw Heatmap (Layer 0)', 'Filtered Heatmap (Layer 0)',
          'Raw Via Probability', 'Boosted Via Probability']
data = [raw_heatmap[0], filtered_heatmap[0], raw_via, boosted_via]
cmaps = ['inferno', 'inferno', 'viridis', 'viridis']

# Color map for nets
net_colors = ['#10B981', '#3B82F6', '#EF4444', '#F59E0B', '#8B5CF6',
              '#EC4899', '#06B6D4', '#84CC16']

for idx, ax in enumerate(axes.flat):
    ax.set_facecolor('#111222')
    im = ax.imshow(data[idx], cmap=cmaps[idx], origin='lower', vmin=0, vmax=1)
    fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    ax.set_title(titles[idx], color='white', fontsize=12)
    ax.set_xticks([])
    ax.set_yticks([])
    
    # Overlay pin locations
    for ni, net in enumerate(board.nets):
        color = net_colors[ni % len(net_colors)]
        is_current = (net.id == current_net.id)
        for pid in net.pin_ids:
            pin = board.pins[pid]
            marker = '*' if is_current else 'o'
            size = 120 if is_current else 60
            edge = 'white' if is_current else '#666'
            ax.scatter(pin.global_x, pin.global_y, marker=marker, s=size,
                      c=color, edgecolors=edge, linewidth=1.5, zorder=10)

# Add legend
from matplotlib.lines import Line2D
legend_elements = []
for ni, net in enumerate(board.nets):
    color = net_colors[ni % len(net_colors)]
    marker = '*' if net.id == current_net.id else 'o'
    label = f'Net {net.id} ({"CURRENT" if net.id == current_net.id else "other"})'
    legend_elements.append(Line2D([0], [0], marker=marker, color='w', markerfacecolor=color,
                                  markersize=10, label=label, linestyle='None'))

fig.legend(handles=legend_elements, loc='lower center', ncol=len(board.nets),
           fontsize=10, facecolor='#1a1a2e', edgecolor='#333',
           labelcolor='white', framealpha=0.9)

plt.tight_layout(rect=[0, 0.06, 1, 0.96])
plt.show()

## 5. End-to-End: Full Routing Step with Filtering

Runs a complete `step_with_heatmaps` and shows whether A* avoids other-net pads.

In [ ]:
from pcb_router.env.pcb_env import PCBRoutingEnv

env = PCBRoutingEnv(board_config=bc)
obs, info = env.reset(seed=42)

print(f'Board: {env.board.width}x{env.board.height}')
print(f'Nets: {len(env.board.nets)}')
print(f'Unrouted: {obs["num_unrouted"]}')

In [ ]:
# Route net 0 with a uniform heatmap - the pin mask should do the heavy lifting
net_idx = 0
uniform_heatmaps = np.ones((env.board.num_layers, env.H, env.W), dtype=np.float32) * 0.5
uniform_via = np.ones((env.H, env.W), dtype=np.float32) * 0.3

next_obs, reward, terminated, truncated, next_info = env.step_with_heatmaps(
    net_idx, uniform_heatmaps, uniform_via
)

connected = next_info.get('connected', False)
path = next_info.get('path', [])

print(f'Net {net_idx}: connected={connected}, reward={reward:.3f}')
print(f'Path length: {len(path)} waypoints')

if path:
    # Check if path passes through any other-net pad locations
    other_pad_locs = set()
    for net in env.board.nets:
        if net.id != env.board.nets[net_idx].id:
            for pid in net.pin_ids:
                pin = env.board.pins[pid]
                for dy in range(-5, 6):
                    for dx in range(-5, 6):
                        if dx*dx + dy*dy <= 25:
                            other_pad_locs.add((pin.global_x + dx, pin.global_y + dy))
    
    violations = []
    for wp in path:
        if (wp[0], wp[1]) in other_pad_locs:
            violations.append(wp)
    
    if violations:
        print(f'WARNING: Path passes near {len(violations)} cells close to other-net pads!')
        for v in violations[:5]:
            print(f'  Waypoint ({v[0]}, {v[1]}, layer={v[2]})')
    else:
        print('Path avoids all other-net pad locations!')

In [ ]:
# Visualize the path over the board
if path:
    fig, ax = plt.subplots(figsize=(10, 10))
    fig.patch.set_facecolor('#111222')
    ax.set_facecolor('#111222')
    
    # Show filtered heatmap as background
    env.board_state.set_current_net(env.board.nets[net_idx].id)
    display_heatmap = uniform_heatmaps[0].copy()
    display_heatmap *= env.board_state.get_pin_exclusion_mask(0)
    ax.imshow(display_heatmap, cmap='inferno', origin='lower', alpha=0.6, vmin=0, vmax=1)
    
    # Draw components
    for comp in env.board.components:
        rect = patches.Rectangle((comp.x, comp.y), comp.width, comp.height,
                                 linewidth=1, edgecolor='#4B5563', facecolor='#1F2937', alpha=0.7)
        ax.add_patch(rect)
    
    # Draw all pads
    net_colors = ['#10B981', '#3B82F6', '#EF4444', '#F59E0B', '#8B5CF6',
                  '#EC4899', '#06B6D4', '#84CC16']
    for ni, net in enumerate(env.board.nets):
        color = net_colors[ni % len(net_colors)]
        for pid in net.pin_ids:
            pin = env.board.pins[pid]
            circle = patches.Circle((pin.global_x, pin.global_y), radius=3,
                                    facecolor=color, edgecolor='white', linewidth=1, alpha=0.9, zorder=5)
            ax.add_patch(circle)
            ax.annotate(f'N{net.id}', (pin.global_x + 4, pin.global_y + 4),
                       color=color, fontsize=8, zorder=6)
    
    # Draw path
    xs = [wp[0] for wp in path]
    ys = [wp[1] for wp in path]
    ax.plot(xs, ys, color='#06B6D4', linewidth=2.5, zorder=8, label='Routed path')
    ax.plot(xs[0], ys[0], marker='*', color='#10B981', markersize=15, zorder=9, label='Start')
    ax.plot(xs[-1], ys[-1], marker='s', color='#EF4444', markersize=10, zorder=9, label='End')
    
    ax.set_xlim(0, env.board.width)
    ax.set_ylim(0, env.board.height)
    ax.set_aspect('equal')
    ax.set_title(f'Routed Net {net_idx} with Pin Exclusion Mask Active',
                color='white', fontsize=14)
    ax.legend(facecolor='#1a1a2e', edgecolor='#333', labelcolor='white', fontsize=10)
    ax.set_xticks([])
    ax.set_yticks([])
    plt.tight_layout()
    plt.show()
else:
    print('No path found - nothing to visualize')

## 6. Route All Nets - Check DRC Violations

In [ ]:
# Fresh env
env2 = PCBRoutingEnv(board_config=bc)
obs2, info2 = env2.reset(seed=42)

results = []
for i in range(len(env2.board.nets)):
    hm = np.ones((env2.board.num_layers, env2.H, env2.W), dtype=np.float32) * 0.5
    vp = np.ones((env2.H, env2.W), dtype=np.float32) * 0.3
    
    next_obs, reward, terminated, truncated, next_info = env2.step_with_heatmaps(i, hm, vp)
    results.append({
        'net': i,
        'connected': next_info.get('connected', False),
        'reward': reward,
        'drc': next_info.get('drc_violations', 0)
    })
    print(f'Net {i}: connected={next_info.get("connected", False)}, '
          f'reward={reward:+.3f}, DRC violations={next_info.get("drc_violations", 0)}')

connected_count = sum(1 for r in results if r['connected'])
print(f'\nRouting completion: {connected_count}/{len(env2.board.nets)}')
print(f'Total DRC violations: {results[-1]["drc"] if results else 0}')